# Auto-sklearn baseline runs (one per use case)

Launches `auto-sklearn` against the same train/test splits used by the ritme experiments, via `submit_automl(...)` from `src/launch_automl.py`. The data splits (`data_splits_*/train_val.pkl`, `test.pkl`) are produced by `submit_model(...)` from `src/launch_models.py`; run those once first (or call `ritme split-train-test` directly).

Run from a conda env with `auto-sklearn` installed:
```shell
mamba create -n autosklearn -c conda-forge -c bioconda -c pytorch auto-sklearn matplotlib scikit-learn ipykernel -y
conda activate autosklearn
pip install -e .
```

## Setup

In [ ]:
from src.launch_automl import submit_automl

## Configuration

In [ ]:
# Where ritme writes experiment outputs. The launcher creates this dir
# (relative to the repo root if not absolute) and stores per-experiment
# subfolders + sbatch logs under it. Override for cluster runs, e.g.
# ``"/cluster/work/<lab>/<user>/ritme_runs_v140"``.
LOGS_DIR = "use_cases/ritme_runs/local_automl"

# Cluster account (SLURM ``--account=...``, a.k.a. "share") forwarded
# to every sbatch submission below. ``None`` (default) leaves the flag
# unset and uses the cluster default; set to your own, e.g. ``"my_lab"``.
SLURM_ACCOUNT = None

# Auto-sklearn search budget (seconds). Default 428400 (~119 h) fills the
# 120 h SLURM tier — autoML has no SHAP/bootstrap overhead so it can use
# the full slot. Match this to your ritme TIME_BUDGET_S for an
# equal-time comparison.
TIME_BUDGET_S = 428_400

# Per-job SLURM resources (slurm mode only; ignored when mode="local").
# Leave as None to use submit_automl's defaults (100 CPUs, 4096 MB/CPU).
# Match `src/launch_models.py:SLURM_RESOURCES[(usecase, ritme_model)]`
# if you want apples-to-apples resources against a specific ritme model.
CPUS = None
MEM_PER_CPU_MB = None

## Submit one auto-sklearn job per use case

Edit `RESTRICTED_MODEL` below to match each usecase's ritme winner (the accompanying manuscript's "ritme vs. generic AutoML" section — e.g., ritme `xgb` ⇄ auto-sklearn `gradient_boosting`, ritme `rf` ⇄ `random_forest`). Valid names live in `src/generic_automl.py:REGRESSION_MODELS` and `CLASSIFICATION_MODELS`; the task (regressor vs classifier) is read from `USECASES[usecase]["task"]` so `mlp` / `random_forest` / `gradient_boosting` work for both.

In [ ]:
# Per-usecase auto-sklearn estimator. Set to match the ritme model class
# that won n2 for each usecase. Default "mlp" is a placeholder until
# the n2 winners are known.
RESTRICTED_MODEL = {
    "u1": "mlp",
    "u2": "mlp",
    "u3": "mlp",
}

common = dict(
    mode="slurm",
    logs_dir=LOGS_DIR,
    slurm_account=SLURM_ACCOUNT,
    total_time_s=TIME_BUDGET_S,
    cpus=CPUS,
    mem_per_cpu_mb=MEM_PER_CPU_MB,
)
for usecase in ("u1", "u2", "u3"):
    submit_automl(usecase, restricted_model=RESTRICTED_MODEL[usecase], **common)